# 第14回: 接線と判別式 — 「接する」とき交点は何個か（二千年かかった言語化）

**主題**: L4 から何度も使ってきた **接線**。L12 の証明の芯（球への二接線は等長）も、L13 の準線も、接線の上に立っていた。だが「**接する**」とは何か、を正面から問うたことはまだない。今日、円と接線の交点を GeoGebra 自身に **数えさせる** と、奇妙な答えが返ってくる。実は Euclid も、この「接する」を証明するとき奇妙な言い方をしている ——「そこに線分は入り込めない」。二千年後、Descartes/Fermat の代数がようやくこれを言い切る言葉を作った。**重解**（同じ場所に重なった二つの解）と **判別式** $D$ である。高校で覚えた「$D = 0 \Leftrightarrow$ 接する」は、公式ではなく、**二千年かかった言語化の到達点** だった。

**副題**: 今日の道具立ては「**数えさせる → ずれに気づく → 式で言い切る**」。作図（GeoGebra）の返す個数と、方程式（SymPy）の返す個数が **合わない** 瞬間を、ごまかさずに見る。道具のずれは道具の恥ではない —— ずれの正体を言葉にできたとき、概念（重解・多重度）が手に入る。もう一つ、L13 で研いだ視点（**変わる量の中の変わらない量**）を、今日は接線で使う。

**学習目標**

1. 外の点 $P$ から円への接線を Thales の円で作図し（L4 の再演・枕の回収）、二接線の **等長** $|PT_1| = |PT_2|$ を確かめられるようになる。
2. 円と接線の交点を GeoGebra に求めさせ、**表示される個数と方程式の解の個数が合わない** ことを観察し、その正体を SymPy の **判別式** $D$ と **重解**（多重度 2）として言い切れるようになる。
3. 直線の傾き $m$ を動かして $D > 0$（2 点で交わる）/ $D = 0$（接する）/ $D < 0$（実の交点なし）の三つの場合を **全部** 列挙し、$D<0$ でも解が（複素数として）消えていないことを確かめられるようになる。
4. $P$ をドラッグして、**変わる量**（接線の長さ）と **変わらない量**（$|OT| = r$、$\angle OTP = 90°$）を区別できるようになる。さらに $P$ を円の **中** に入れても生き残る直線（極線）に出会う。

```{admonition} 目標の確認 — @Codex で達成目標を引く
:class: note
本回の達成目標を **@Codex に lancedb-rag（教材ドメイン RAG）で確認**してもらえる（L5 から運用）。
（プロンプト例）@Codex この回（L14 接線と判別式）の達成目標を lancedb-rag で調べて、要点を整理して。
特に「GeoGebra の交点の個数と方程式の解の個数が合わない」「接する = 重解」「D の三つの場合を全部見る」が
腑に落ちるか、自分の理解と照らしてから先へ進む。
```

## 0. 前回 (第 13 回) の振り返り — 全員が立体に接地した。だが「接する」が残った

前回、準線が接触円の平面と切断面の交わりとして立体から現れ、離心率 $e$ が三曲線を貫いた。L10 から始めた円錐曲線の登場人物は全員、立体に接地した。

その全部を下から支えていた言葉が「接する」である。球は切断面に **接し**、円錐に **接し**、$P$ からの線分は球に **接して** いた。L4 では円の接線を作図し、二接線の等長を使った。作図としての接線は、もう手の中にある。

> **今日の問い**: 円と接線が「接する」とき、**交点は何個か**。二個の交点（交わる）と零個の交点（離れる）のあいだを、「一個」はどうやって通り抜けるのか。

まず、枕（L13 末尾）の作図を呼び戻すところから。

## 1. ggblab セットアップと接線の作図（枕の回収）

In [ ]:
using Pkg
Pkg.activate("../..")   # この教材プロジェクトの環境を有効化
Pkg.resolve()
Pkg.instantiate()
using GeoGebra
ENV["GGB_DIRECT_TRANSPORT"] = "true"

In [ ]:
inject_applet()

In [ ]:
# L13 の枕と同じ: 外の点 P から円 c への接線を Thales の円で引く（今日は 2D）。
# 規律1: 各オブジェクトを一意のシンボルに束縛し、入れ子にしない。
@ggb :const :new
@ggb O=(0, 0)
@ggb c=Circle(:O, 2)                       # 半径 2 の円
@ggb P=(4, 0)                              # 円の外の点（あとでドラッグする）
@ggb M=Midpoint(:O, :P)
@ggb th=Circle(:M, :O)                     # Thales の円（OP を直径とする円）
@ggb l_t="{Intersect(c, th)}"
@ggb T_1=l_t(1)
@ggb T_2=l_t(2)                            # 二つの接点
@ggb t_1=Line(:P, :T_1)
@ggb t_2=Line(:P, :T_2)                    # 二本の接線

**観察1**: なぜ Thales の円でうまくいくのか —— `th` の上の点から見ると、直径 $OP$ は **直角** に見える（L4 の公準5「半円に内接する角は直角」）。だから交点 $T_1$ では $\angle OT_1P = 90°$、つまり半径 $OT_1$ と直線 $PT_1$ が直交する。「半径と直交する直線は接線」—— L4 の道具だけで接線が引けた。

In [ ]:
# L4/L12 の「二接線の等長」を数値でも確認。ついでに直角と半径も。
@ggb len_1=Distance(:P, :T_1)
@ggb len_2=Distance(:P, :T_2)              # len_1 = len_2 = 2√3 ≈ 3.4641
@ggb ang=Angle(:O, :T_1, :P)               # 90° になるはず
@ggb rad=Distance(:O, :T_1)                # = 2（接点は円の上）

**観察2**: $|PT_1| = |PT_2| = 2\sqrt{3} \approx 3.4641$（等長）、$\angle OT_1P = 90°$、$|OT_1| = 2$。L12 で球に持ち上げたあの事実の、2D の原型が全部そろった。

## 2. GeoGebra に交点を数えさせる —— 奇妙な答え

### 2.1 予測 —— 接線と円の交点は何個のはずか

```{admonition} 予測を書き留める —— 数えさせる前に
:class: important
接線 `t_1` と円 `c` の交点は **何個** のはずか。比較のため、円を貫く直線（割線）なら何個か、
円から離れた直線なら何個か。三つの場合の予測を書き留めてから、次のセルへ進む。
```

### 2.2 確認 —— 三種類の直線で数えさせる

In [ ]:
# 円 c と 三種類の直線（割線・接線・離れた直線）の交点を、GeoGebra に求めさせる。
@ggb l_{sec}=Line(:P, (0, 1))                # 割線（円を貫く直線）
@ggb l_{far}=Line((4, 3), (0, 3))            # 離れた直線 y = 3（円に届かない）
@ggb i_s="{Intersect(c, l_{sec})}"           # 割線との交点（2 個返るはず）
@ggb i_t="{Intersect(c, t_1)}"             # 接線との交点（さて何が返るか）
@ggb i_f="{Intersect(c, l_{far})}"           # 離れた直線との交点（さて何が返るか）

In [ ]:
# それぞれのリストの中身と「長さ」を見る。
@ggb n_s=Length(:i_s)
@ggb n_t=Length(:i_t)
@ggb n_f=Length(:i_f)

**観察3**: 割線は素直に 2 点が返る。ところが接線と離れた直線では、様子がおかしい —— 返ってきたリストの中に **「定義されていない」点（座標が表示されない・undefined）が混じる**。GeoGebra は「交点用の席」を二つ用意した上で、実際に置けなかった席を undefined のまま返してくる。だから `Length` で数えると、**目で見た個数と合わない** ことが起こる。接点は目には一個なのに、席は二つある。離れた直線は交点が零個のはずなのに、undefined の席が数に紛れ込むことがある。

**これは GeoGebra のバグではない**。GeoGebra は「円と直線の交点は最大 2 個」という **方程式の事情** を知っていて、席を常に 2 つ作る。実際に座れるかどうかは数値計算の結果次第で、座れなかった席が undefined になる。つまりこの奇妙な表示は、**作図の見た目（1 個に見える）と方程式の答え（2 つある）のずれ** が画面に露出したものである。ずれの正体を、式の側から言い切りに行く。

## 3. SymPy で言い切る —— 重解と判別式

### 3.1 円と直線を連立する

$P = (4, 0)$ を通る傾き $m$ の直線 $y = m(x - 4)$ を、円 $x^2 + y^2 = 4$ に代入すると、$x$ の二次方程式になる。L11 と同じ作法（Julia から SymPy を借りる）で解く。

In [ ]:
# L11 と同じ作法（Julia から Python の SymPy を借りる）。
using PythonCall
sympy = pyimport("sympy")
x, m = sympy.symbols("x m", real=true)
expr = x^2 + (m*(x-4))^2 - 4               # 円 x²+y²=4 に y=m(x−4) を代入したもの
poly = sympy.expand(expr)
println(poly)                               # (m²+1)x² − 8m²x + 16m² − 4

In [ ]:
# 二次方程式の判別式 D を SymPy に計算させ、D = 0 になる傾き m を求める。
D = sympy.discriminant(poly, x)
println("D = ", sympy.factor(D))            # −16(3m² − 1)
println("D = 0 → m = ", sympy.solve(sympy.Eq(D, 0), m))   # m = ±√3/3 = ±1/√3

**観察4**: $D = -16(3m^2 - 1)$、$D = 0$ となるのは $m = \pm\frac{\sqrt{3}}{3} = \pm\frac{1}{\sqrt{3}}$。作図の接点 $T_1 = (1, \sqrt{3})$ と $P = (4,0)$ を結ぶ傾きは $\frac{\sqrt{3} - 0}{1 - 4} = -\frac{1}{\sqrt{3}}$ —— **作図で引いた接線の傾きが、式の $D = 0$ からぴったり出てきた**。

### 3.2 接するとき、解は「一個」ではない —— 重解

$D = 0$ の傾きで、解そのものを見る。

In [ ]:
# 接線の傾き m = −1/√3 のときの解を、多重度ごと全部並べる。
m0 = -1/sympy.sqrt(3)
p0 = sympy.Poly(expr.subs(m, m0), x)
println("解(多重度つき): ", p0.all_roots())     # [1, 1] — x = 1 が「二回」

**観察5**: 解は `[1, 1]` —— $x = 1$ が **二回** 出てくる。解が一個になったのではない。**二つの解が同じ場所に重なった** のである。これを **重解**（多重度 2 の解）という。§2 の undefined の席の正体はこれだ —— 席が二つあるのは方程式に解が（重なってでも）二つあるからで、GeoGebra は重なった二つを一つの数値でしか置けず、残りの席が undefined になる。**「接する」とは、交点が一個になることではなく、二つの交点が一点に重なること** —— 交わる（2 個）から離れる（0 個）への通り道が、この「重なる一瞬」である。

### 3.3 三つの場合を全部見る —— $D$ の符号と、消えない解

In [ ]:
# 傾き m を三通りに振って、解を全部（複素数まで）見る。場合を一つも捨てない。
for (mv, label) in [(sympy.Rational(3,10), "D>0 交わる"),
                    (-1/sympy.sqrt(3),     "D=0 接する"),
                    (sympy.Rational(4,5),  "D<0 離れる")]
    pv = sympy.Poly(expr.subs(m, mv), x)
    println(label, ":  解 = ", [sympy.N(r, 5) for r in pv.all_roots()])
end

**観察6**: 三つの場合が一枚の表になる。

| 場合 | $D$ | 目に見える交点 | 方程式の解 |
|---|---|---|---|
| 交わる（割線） | $D > 0$ | 2 個 | 実数解 2 個（$-1.2374$ と $1.8980$） |
| 接する（接線） | $D = 0$ | 1 個に **見える** | **重解**（$x=1$ が多重度 2） |
| 離れる | $D < 0$ | 0 個 | **複素解 2 個**（$1.561 \pm 1.170\,i$）—— 消えていない |

$D < 0$ で解は消えるのではなく、実数の世界から **見えなくなる** だけで、複素数としては二つとも生きている。どの場合でも解は（重なりや複素数を数えれば）**きっちり 2 個** —— 「円と直線の交点は最大 2 個」の「最大」の内訳が、これで全部言えた。GeoGebra が用意していた「二つの席」は、この 2 個のためのものだった。

## 4. 二千年前のごまかしと、二千年後の言語化

### 4.1 Euclid は「接する」をどう言ったか —— 原論で唯一、ごまかした場所

Euclid『原論』第 III 巻 命題 16 は、円の直径の端で直径に直交する直線が円の外に落ちることを示す —— 今日の言葉で「接線」である。その証明の言い回しが独特だ:

> この直線と円周との **あいだに、別の直線は入り込めない**。

交点が何個とは言わない。「重なった二つの解」とはもちろん言わない。**隙間のなさ** で接することを言い表す。そしてこの命題には続きがある —— 接線と円周のあいだにできる「角」（**ホーン角**・角のように見えて、どんな直線の角よりも小さいもの）は、原論全体の中で **量の比較のルールが公理系からはみ出す唯一の箇所** として知られる。465 の命題を積み上げた大伽藍の中で、接線の足元のこの一点だけ、Euclid は量として正面から扱うことを避けた。**正直な行き詰まりの表示** と読むこともできるし、あえて厳しく言えば、原論が **言論の中で唯一ごまかした** 箇所である。§2 で GeoGebra が見せたずれと、Euclid のこの一点は、二千年を隔てて **同じ場所** に立っている。

### 4.2 Apollonius は接線を作図した —— そして問いを封印した（通径の二つの顔）

Apollonius『Conica』は円錐曲線の接線を **正確に作図する** 方法を持っていた。その体系の背骨が **通径**（latus rectum）—— 曲線一本を一つの量で記述する、関数概念の遠い先祖であり、実は今も $\ell = a(1-e^2)$ という形で軌道力学や GPS の中で **現役** の量である。だが Apollonius の接線の定義は、本質的に Euclid と同じ **静的なもの**（曲線と一点でしか交わらない直線）であり、通径の体系はその静的接線の上に建っている。「なぜその直線が接するのか」「接する瞬間に何が起こっているのか」という **局所の問い** は、比例と面積の言葉の中に折り畳まれ、体系の外からは見えない。

つまり通径は二つの顔を持つ —— **後世への贈り物**（曲線を量で記述する初の体系）と、**封印の道具**（接線の局所構造への問い、すなわち後の微分につながる問いを、問わずに済ませる装置）。両者は表裏一体である。L13 §5.2 と同じく、これは **診断（読み）であって、証明された因果ではない** —— 「封印のせいで千八百年遅れた」とまで言い切れるかは、実は検証しにくい（この「言い切れる/言い切れない」の線引きは、皆さんの最終レポートでもそのまま問われる）。確かなのは、接線の局所構造が厳密な言葉（極限）を得るまでに、ここから千八百年以上かかったという **年表の事実** の方である。

### 4.3 Descartes と Fermat —— 二つの脱出路と、一つの論争

1637 年前後、Descartes と Fermat が曲線を **方程式** に写した（L11 で再演した画期）。曲線と直線の交わりが「連立方程式の解」になった瞬間、交点は **数えられるもの** になった。だが二人の接線への道は対照的だった:

- **Descartes は法線から行った**。曲線に直交する直線（法線）を、扱える概念だけで代数的に構成し、**極限という未定義概念には踏み込まない**。誠実な理論化 —— 未定義なものを使わないという意味で、実は Apollonius の精神の正統な継承者でもある。
- **Fermat は極限を引き受けた**。接線を「二つの交点が重なる極限」として扱う方法（**adequality**）を編み出し、封印を **直接破壊** する方向に進んだ。Descartes はこれを「未定義概念に依存している」と批判した —— 批判として筋は通っている。
- **歴史が選んだのは Fermat の方向** である。危うい方の道が微分になった。そして **Newton** は、この論争の決着を **保留したまま**、極限もどきの道具（ultimate ratios）を **運用レベル** で物理に投入し、『Principia』で力学を立ち上げた。Berkeley 司教に「消えた量の幽霊」と揶揄され、十分に答えないまま、応用で押し切った —— 厳密な決着（Cauchy/Weierstrass の極限の定義）は、さらに百五十年後である。

高校で覚えた

$$D = 0 \iff \text{接する}$$

は、この文脈に置くと公式暗記ではなくなる —— **Euclid が「入り込めない」としか言えなかったもの、Apollonius が問わずに済ませたもの、Descartes が迂回し Fermat が踏み込んだもの** を、誰でも数えられる形にした到達点である。そして「厳密には未決着のまま、運用で二百年前進した」という Newton の選択は、正しさと前進が **同じ速度では進まない** ことの、科学史上最大級の実例である。

```{admonition} なぜ「ずれ」から始めたのか
:class: note
今日の入口は GeoGebra の undefined だった。道具が見せる小さなずれは、しばしば **概念の不在** が
露出した場所である。GeoGebra のずれ（席は二つ、見た目は一つ）は、Euclid のごまかし（入り込めない）と
**同じ場所** に立っていた —— どちらも「重なった二つ」を言う言葉の問題である。道具のずれを
「そういうものか」で流さずに正体を問うと、二千年ものの概念に手が届くことがある。
```

## 5. 動かして視る —— 変わる量、変わらない量、そして生き残る直線

### 5.1 ドラッグ —— L13 の視点を接線で使う

In [ ]:
# P をドラッグしながら、次の値を見る: len_1（接線の長さ）/ rad（|OT1|）/ ang（∠OT1P）。
# どれが変わり、どれが変わらないか。—— 動かす前に予測を書き留めること。
@ggb pow="len_1^2 + rad^2"                   # おまけ: |PT1|² + r² は何と等しいか（|OP|² と比べよ）
@ggb dOP=Distance(:O, :P)

**観察7**: $P$ を（円の外で）どこへ動かしても —— **変わる**: 接線の長さ $|PT_1|$。**変わらない**: $|OT_1| = 2$（接点は常に円の上）と $\angle OT_1P = 90°$（半径と接線の直交）。そして $|PT_1|^2 + r^2 = |OP|^2$ が常に成り立つ（三平方。$|PT_1|^2 = |OP|^2 - r^2$ を **円の冪** と呼ぶ）。L13 では「比」が不変量だった。今日は「長さ $r$」と「直角」が不変量である。**動かして、変わらないものを見つけたら、それがその構成の芯** —— この作法は幾何に限らない。

### 5.2 予測 —— $P$ を円の中へ入れたら

```{admonition} 予測を書き留める —— 境界を越える前に
:class: important
$P$ を円周へ近づけると、二つの接点 $T_1, T_2$ はどうなるか。$P$ が **円周上に来た瞬間** は?
さらに **円の中へ入れたら**? 接線は、接点は、Thales の円はどうなるか。
§3.3 の表（$D$ の三つの場合）と見比べながら、三段階の予測を書き留めてから動かす。
```

**観察8**: $P$ が円周に近づくと $T_1$ と $T_2$ は互いに近づき、円周上で **一点に重なる**（このとき接線は 1 本、$P$ 自身が接点 —— まさに重解の瞬間）。円の中に入ると $T_1, T_2$ は undefined になり、接線は消える。$D = |OP|^2 - r^2$ とおけば、これは §3.3 の表そのものである —— $D > 0$ 外（接線 2 本）/ $D = 0$ 円周上（重なって 1 本）/ $D < 0$ 中（実の接線なし）。

### 5.3 それでも生き残る直線 —— 極線（発展）

$D < 0$ で「すべて消えた」わけではない。複素解が生きていたように、作図の側にも生き残る直線がある。

In [ ]:
# 発展: 極線。P が円の中に入っても消えない直線(GeoGebra の Polar コマンド)。
@ggb pol=Polar(:P, :c)

**観察9**: $P$ が円の **外** にいるとき、`pol` は二つの接点 $T_1, T_2$ を通る直線（接点どうしを結ぶ弦）。$P$ が **円周上** に来ると、`pol` は接線そのものに重なる。そして $P$ を **円の中** に入れても —— 接点も接線も消えたのに —— `pol` は消えずに、円の外側を平行移動しながら生き続ける。境界（円周）を越えても連続に生き残るこの直線は、$D < 0$ の複素解 $1.561 \pm 1.170\,i$ の **実部の世界での影** である（実際、複素の二接点を「結んだ」直線が実直線としてこれになる）。**境界で消えたように見えるものの向こうに、続きがある** —— L11 の複素解、L12 §6 の反対側の円錐（双曲線のもう一枝）、そして今日の極線。同じ型が三度目である。

## 6. 冷間チェックの作法 —— 五年の物語も、冷えてから一撃を受けた（実話）

L13 で出題した学期末レポート（科学史転生無双・最終版）のために、今日の主題（ごまかさずに数える）を **書く側の作法** として渡す。実話から始める。

### 6.1 実話 —— 太字の一行が偽だった

この講義の縦軸（Menaechmus から Feynman までの物語）は、先生が五年をかけて紡ぎ、AI との長い対話で検証しながら一つの文書に総括されたものである。その総括文書が、**書き上げて二ヶ月ほど寝かせたあと、もう一度 AI に「今度は徹底的に粗を探せ」という姿勢で読み直させた** ところ —— 太字で強調された中心的な一行が **数学的に偽** であることが見つかった。

その一行とは: 「内心と傍心は、三角形の辺について対称的に配置される」。L8–L9 でやった、あの話である。もっともらしく聞こえる。だが偽である —— 確かめてみよう。

In [ ]:
# 反例は GeoGebra 数分。不等辺三角形で、内心 I を辺 BC で折り返すと傍心 I_A に重なるか?
@ggb :const :new
@ggb A=(1, 2.5)
@ggb B=(0, 0)
@ggb C=(5, 0)                              # BC は x 軸上（折り返しが y 反転で見やすい）
@ggb tri=Polygon(:A, :B, :C)
@ggb I=TriangleCenter(:A, :B, :C, 1)       # 内心（Kimberling X(1)）
@ggb l_{BC}=Line(:B, :C)
@ggb I_{ref}=Reflect(:I, :l_{BC})              # 内心を辺 BC で折り返した点

In [ ]:
# 傍心 I_A（A の対辺 BC 側の傍接円の中心）を外角二等分線から作る（L8 の作図）。
# I_ref と I_A が一致するかを Distance で判定する。
@ggb w_B=AngleBisector(:A, :B, :C)
@ggb w_C=AngleBisector(:A, :C, :B)
@ggb w_Bx=PerpendicularLine(:B, :w_B)      # B での外角二等分線（内角二等分線に直交）
@ggb w_Cx=PerpendicularLine(:C, :w_C)
@ggb I_A=Intersect(:w_Bx, :w_Cx)
@ggb gap=Distance(:I_ref, :I_A)            # 0 なら対称。さて。

**観察10**: `gap` は 0 にならない（この三角形では $I \approx (1.49, 1.01)$、折り返しは $(1.49, -1.01)$、だが $I_A \approx (3.51, -5.19)$ —— まるで別の場所である）。内接円の半径 $r$ と傍接円の半径 $r_A$ は **常に** $r_A > r$ なので、どんな不等辺三角形でも中心同士は鏡映で重ならない。**では L9 で発見した対称は嘘だったのか? —— 嘘ではない。対称だったのは中心ではなく「接点」である**（L9 の測れる命題そのもの: $BD = s-b$, $BD' = s-c$, 和 $= a$、ゆえに中点対称）。中心同士の関係は「内角/外角の二等分」という **双子（双対）** であって、鏡映対称ではない。—— 総括文書は、正しい接点の対称（手順の中では正しく書かれていた）を、太字の要約で「中心の対称」へと **言いすぎて** いた。

### 6.2 この実話から取り出す作法

1. **書いた直後の自分は、一番甘い読者である**。書き上げた熱（うまく繋がった! という感覚）は、要約や太字でこそ「言いすぎ」を生む。冷ましてから読み直す —— これを **冷間チェック** と呼ぶ。
2. **粗探しは「一番痛い一撃」を探す**。細かい指摘を並べる（あら探しのリスト）のではなく、「これが崩れたら主張全体が割り引かれる急所はどこか」を一つ選んで突く。急所が生き残れば主張は強い。
3. **修復すると、たいてい前より強くなる**。この実話でも、偽の「中心対称」を正しい「接点対称」に置き換えたら、証明は L6 の「二接線は等長」だけで落ちるようになり（= Dandelin 証明と同じ一手）、物語はむしろ締まった。**誤りの発見は物語の敗北ではなく、物語が本物になる工程** である。
4. **AI は讃めさせるな、突かせろ**。同じ AI が、書く伴走では甘く、「反証を探せ」と役を与え直せば鋭くなる。役の与え方が出力の質を決める。

```{admonition} 最終レポートの中間チェック（今週の宿題）
:class: important
L13 で出題した転生無双レポートの下書き（部分でよい）に、この作法を適用する:
1. 下書きを **一日以上** 寝かせる。
2. `@Codex` に **役を与えて** 読ませる —— 「あなたは私のレポートの一番痛い一撃を探す係。太字・要約・
   言い切りの文を優先して、事実として偽の箇所、言いすぎの箇所を **一つだけ** 選んで突いて」。
3. 一撃が当たったら、**主張を弱めるのでなく、正確に言い直す**（中心対称 → 接点対称、のように）。
   当たらなかったら、その一撃と防御をレポートの脚注に残す（反証条件の実演として評価対象になる）。
```

```{admonition} 進捗の確認 — セル出力を @Codex に読ませる
:class: tip
@Codex は **jupyter-server-mcp であなたのセル入出力を直接読みます**（プロンプトだけではない）。
（プロンプト例）@Codex ここまでのセル出力を読んで、達成目標①（Thales 接線と等長）②（GeoGebra の
個数のずれを観察した）③（D の三つの場合と重解・複素解）④（変わる量/変わらない量の区別と極線）に
どこまで到達したか、まだ埋まっていないセルはどこか、具体的に挙げて。
```

```{admonition} 今回の課題
:class: tip

**必修**
1. Thales の円で外点 $P$ からの二接線を作図し、等長 $|PT_1| = |PT_2|$・直角 $\angle OT_1P = 90°$・円の冪 $|PT_1|^2 = |OP|^2 - r^2$ を確かめよ（§1・§5.1）。
2. 円と「割線・接線・離れた直線」の交点を GeoGebra に求めさせ、undefined の席を観察せよ（§2）。同じ三つの場合を SymPy の判別式と `all_roots`（多重度・複素解込み）で言い切り、観察6 の表を自分のノートで完成させよ（§3）。

**思考課題**
3. Euclid III.16 の「あいだに直線は入り込めない」という言い方と、「重解（多重度 2）」という言い方が、**同じ現象のどの面** をそれぞれ捉えているかを論ぜよ。Euclid に足りなかったのは頭の良さか、それとも **記法**（代数の言葉）か。
4. L12 §3.4 の「球への二接線は等長」と、今日の「円への二接線は等長」は同じ scope の違う実装だった。では今日の **判別式** $D = |OP|^2 - r^2$ に対応するものを球で書くとどうなるか。さらに L13 の離心率も「二つの量の比」だった —— 円錐曲線の話には、なぜこうも「二つ」が繰り返し現れるのか、現時点の見立てを書き留めよ（転生無双レポート・L15 議論の素材）。
5. **(発見してほしい問い)** §5.3 の極線 `pol` について: $P$ を動かしながら、$O$ から `pol` までの距離と $|OP|$ を測り、二つの積を計算せよ。何が見えるか。その「変わらない量」は、今日のどの式と同じものか（ヒント: 円の冪）。
```

:::{important} 授業末尾の自己評価 —— `@Codex` に聞いてみる（任意、L4–L13 から継続）

L4–L13 と同じ template で、本回の自己評価を試してください。**任意**です。

````text
@Codex 今日のノートブックを全 cell 読んで評価してください。
次の三つを区別して articulate してください:

1. 自分で考えて書いた cell — 思考の痕跡が残っている部分
2. AI 委託で書いたが、理解して受け入れた cell — 動いて、なぜ動くか説明できる部分
3. AI 委託で書いたが、なぜ動くか説明できない cell — 動いているが、理解で未到達の部分

加えて: 今日の主題（GeoGebra の個数のずれ / 接する = 重解 / D の三つの場合と複素解 /
変わる量と変わらない量 / 極線が境界を越えて生き残る）に対する到達度、
完成しないまま残った問い、次回（L15 最終回・Feynman と発表会）への接続点。

特に「D=0 ⇔ 接する を公式として覚えていたときの理解」と「重解として腑に落ちた今の理解」の
差を、誤魔化さず正直に articulate してください。
最終判断は自分で。@Codex の照合は候補の提示であって、評価は自分の構成(construction)に対して行う。
````

JupyterAI のやり取り log は LMS 経由で先生に届きます —— 学期末レポート・発表（第 15 回・次回）の materials として毎週蓄積。
:::

## 7. 次回への接続

今日、「接する」が言葉になった —— 二つの交点が一点に重なること。重なりを数える言葉（重解・判別式）は二千年かかって届いた。境界を越えても生き残るもの（複素解・極線）を三度目に見た。そして、五年の物語でさえ冷間チェックで強くなるという実話とともに、レポートを磨く作法を渡した。

最終回（第 15 回）は、接線を **一本ずつ** ではなく **束** として見る。ある円と一点から、垂直二等分線を **たくさん** 引くと —— 一本一本はただの直線なのに、束が **浮かび上がらせる曲線** がある。その曲線は、この学期の主役だった楕円である。そしてこの見方こそ、1964 年に Feynman が Newton の重力法則から惑星の楕円軌道を **微積分を使わず初等幾何だけで** 導いたときの道具だった。Dandelin（1822）に続く「二千年の見落とし」のもう一例を最終回の入口に置き、後半は **皆さんの転生無双レポートの発表と議論** —— この講義全体の問い「画期はいかに起こり、停滞はなぜ続くか」に、皆さん自身の声で答える時間である。冷間チェックを通した下書きを持ってくること。

## 枕（次回への予告）—— 一本の垂直二等分線

In [ ]:
# 次回の枕: 円 lk の上の点 Q と、円の中の点 F との垂直二等分線を一本だけ引く。
# 次回、この直線を「束」にしたとき何が浮かび上がるかを予測してもらう。
@ggb :const :new
@ggb F_2=(0, 0)
@ggb lk=Circle(:F_2, 4)                    # 大きな円（半径 4）
@ggb F_1=(1.5, 0)                          # 円の中の点
@ggb Q=Point(:lk)                          # 円の上の点（ドラッグできる）
@ggb m=PerpendicularBisector(:F_1, :Q)     # F1 と Q の垂直二等分線

**読み方**: $Q$ をドラッグすると、直線 `m` が向きを変えながら動く。一本では何も見えない。だが、$Q$ を円の上で一周させたときに `m` が **掃いていく跡** を目で追ってみてほしい —— 何かが囲われていく感じがしないだろうか。予測を書き留めてから、次回。

## 参考文献

- [Euclid's Elements, Book III, Proposition 16](https://mathcs.clarku.edu/~djoyce/java/elements/bookIII/propIII16.html)（「あいだに直線は入り込めない」。Heath 訳での接線の扱い。命題 17 が接線の作図）
- [Tangent lines to circles - Wikipedia](https://en.wikipedia.org/wiki/Tangent_lines_to_circles)（外部点からの二接線・等長・Thales の円による作図・円の冪）
- [Discriminant - Wikipedia](https://en.wikipedia.org/wiki/Discriminant)（判別式。$D=0$ と重根）
- [Multiplicity (mathematics) - Wikipedia](https://en.wikipedia.org/wiki/Multiplicity_(mathematics))（重解・多重度。「解の個数」を過不足なく数える言葉）
- [Pole and polar - Wikipedia](https://en.wikipedia.org/wiki/Pole_and_polar)（極線。P が円内でも定義され、円周上で接線に一致する）
- [Adequality - Wikipedia](https://en.wikipedia.org/wiki/Adequality)（Fermat の接線法。「二交点が重なる」扱いの原型、微分の入口）
- 前回 第13回（準線と離心率）は @Codex に lancedb-rag で「第13回 準線 離心率 接触円 比一定」を尋ねる（project=textbook）
- 第4回（Thales・円の接線）は @Codex に lancedb-rag で「第4回 Thales 半円 直角 接線」を尋ねる（project=textbook）
- 第11回（SymPy の作法）は @Codex に lancedb-rag で「第11回 SymPy PythonCall 方程式」を尋ねる（project=textbook）
- GeoGebra の接線交点が NaN スロットを返す現象の背景は @Codex に lancedb-rag で「接線 NaN 交点 重解 判別式 Euclid ごまかし」を尋ねる（project=conversations）
- §4 の歴史叙述（ホーン角・通径の二重性・法線 vs adequality・Newton の運用統合）の原典は @Codex に lancedb-rag で「Apollonius 通径 接線の封印 Descartes Fermat adequality 数学史対話」を尋ねる（project=conversations） —— 先生五年の物語の総括（2026-05-11 discussion_memo §4）
- §6 の実話（冷間再スイープ・接点対称への修復）は @Codex に lancedb-rag で「discussion_memo 冷間再スイープ 致命傷 内心 傍心 接点 中点対称」を尋ねる（project=conversations）（2026-07-03 REVIEW）

```{admonition} 著者ノート —— 数値検証と実機確認事項（学生用ではない）
:class: note

**数値は SymPy で事前検証済み**（本 session・2026-07-06）: 判別式 $D = -16(3m^2-1)$ / $D=0 \to m = \pm\sqrt{3}/3$ /
重解 `all_roots = [1, 1]` / $m=3/10$ で実解 $\{-1.2374, 1.8980\}$ / $m=4/5$ で複素対 $1.561 \pm 1.1697\,i$ /
接点 $(1, \pm\sqrt{3})$・接線長 $2\sqrt{3} = 3.4641$。円の冪・極線の距離積 $d(O,\text{pol}) \cdot |OP| = r^2$（思考課題5の答え）。

**discussion_memo の埋め込み（2026-07-06 改版）**: §4.1 ホーン角（原論で唯一 量の比較が公理系から逸脱・
「言論の中で唯一誤魔化した」）・§4.2 通径の二重性（贈り物 $\ell = a(1-e^2)$ 現役 / 封印の道具）・§4.3
法線=誠実な理論化 vs adequality=直接破壊・Descartes の Fermat 批判・歴史的勝利は Fermat 方向・Newton
保留+運用+Berkeley = discussion_memo §4 の教材化。「診断であって因果の証明ではない」の register は冷間
スイープ F1 の反映（封印テーゼを因果事実として書かない）。§6 全体 = 冷間スイープ REVIEW（2026-07-03）の
教材化: 実話（§1 致命傷 = 中心対称は偽・接点対称が真・$r_A > r$）+ 単一致命傷規律（§6.2 作法 2）+
「修復後はむしろ強くなる」（REVIEW §1.3）。反例数値は本 session で検証済み（$I=(1.4878, 1.0073)$,
$I_A=(3.5122, -5.1876)$, $BX=1.4878$, $BX_A=3.5122$, 和 $=5=a$）。L8/L12 の緩い「中心が辺対称」文言は
接点版に修正済み（L12 は 2026-07-06 修復、L9 はもともと接点版で正しい）。

**実機確認が必要な点（授業前・G-C gate）**: (1) `{Intersect(c, t_1)}`（接線）と `{Intersect(c, l_far)}`（離れた直線）の
**undefined スロットの正確な表示**（座標表示・`Length` の返り値が版によって揺れる可能性。観察3 の記述は
実機の挙動に合わせて微調整すること。現象自体は eg9 系の既知挙動 = 「交わらないのに個数が合わない」）。
(2) `Polar(点, 円)` コマンドの名称（日本語 UI では「極線」。classic 6 で `Polar` が通ることを確認）。
(3) `Angle(O, T_1, P)` の返り値の単位（度 / ラジアン設定）。
(4) §6 の `TriangleCenter(A,B,C,1)` と `Reflect(点, 直線)`・外角二等分線の `PerpendicularLine(B, w_B)` 構成。

**antidescartes との対応（研究側 register・本文には出さない）**: 本回は `antidescartes/tangent_dag.py`
（第二幾何領域・2026-07-06 実装）の教材側出口である。対応表:
観察7 の「変わる量/変わらない量」= drag-invariance の register 対比（`tlen` の振れ幅 vs `dOT1 ≡ r`・`perp1 ≡ 0` の
値の恒等不変量）。§5.2 の三段階 = `TangentClause` の三節分割（OUTSIDE/ON_CIRCLE/INSIDE・境界を畳まない）。
§5.3 の極線 = `pol_d`（境界を連続に貫く継続対象・`continuous_through_boundary` で検証済み）。
観察5 の重解 = `test_on_circle_is_double_root`（$T_1 = T_2$ を重解の幾何的表現として固定）。
作図の原典は ggblab `examples/eg11_slider.ipynb` cell 8（Thales 構成と `{Tangent}` の突き合わせ）。
予測 admonition 三箇所 = predict-then-reveal の平語化。内部用語は本文に出していない（G-C 授業投入は先生 review 後）。
```